In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import umap
import warnings 
warnings.filterwarnings("ignore")

data_path = "/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/FLabBench-pipeline/saved_data/cohorts/DTB/selected_edges_DTB_all.csv"
save_dir = "/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/FLabBench-pipeline/saved_data/plots/DTB/"
os.makedirs(save_dir, exist_ok=True)

## Load and prepare features

In [ ]:
df = pd.read_csv(data_path)

df = df[(df["n_pos"] > 10) & (df["n_neg"] > 50)].copy()

df["n_cohort"] = df["n_pos"] + df["n_neg"]
df["target_rate"] = df["n_pos"] / df["n_cohort"]
df["log_target_rate"] = np.log(df["target_rate"])
df["log_n_cohort"] = np.log(df["n_cohort"])
df["log_RR"] = np.log(df["RR"])
df["female_rate"] = df["female_counts"] / df["counts"]
df["death_rate"] = df["death_counts"] / df["counts"]
df["imbalance_ratio"] = df["n_neg"] / df["n_pos"]
df["log_imb"] = np.log(df["imbalance_ratio"])
#df = df [df["target_rate"] > 0.01]

p = df["target_rate"]

icd_map = {
    "A": "infectious", "B": "infectious", "C": "neoplasms",
    "D": "blood", "E": "metabolic", "F": "mental",
    "G": "nervous", "H": "sensory", "I": "circulatory",
    "J": "respiratory", "K": "digestive", "L": "skin",
    "M": "musculoskeletal", "N": "genitourinary", "O": "pregnancy",
    "P": "perinatal", "Q": "congenital"
}
df["D1type"] = df["D1"].str[:1].map(icd_map)
df["D2type"] = df["D2"].str[:1].map(icd_map)

print(f"Total valid cohorts: {len(df)}")
df[["n_cohort", "target_rate", "RR", "CODE_DIFF_DAYS"]].describe()

df["cohort_name"] =  "cohort_" + df["D1"] + "_" + df["D2"] + "_" + df["CODE_DIFF_DAYS"].astype(str) 

possible_cohorts = df.copy()

In [ ]:
# Cohorts are heavily right-skewed. Most cohorts are small a few very large
# Target rate imbalance across all cohorts
# PR most cohorts have modest relative risk but sume very strong
# Time window between D1 and D2 mostly consistent (1-2 years)


# For target_rate I didn't use log since the values near zero can result in very large negative numbers and dominate the clustering
# long only for n_cohort and RR

## Cohort Characteristics

In [ ]:
feat_cols = ["target_rate", "log_n_cohort", "log_RR", "AGE_AT_DISEASE", "female_rate", "death_rate", "log_imb", "CODE_DIFF_DAYS"]
possible_cohorts["D1cat"] = possible_cohorts["D1"].str[:1]
possible_cohorts["D2cat"] = possible_cohorts["D2"].str[:1]
cmap = sns.color_palette("husl", 16)

fig, axes = plt.subplots(3, 4, figsize=(20, 12))
axes = axes.ravel()

for ax, feat in zip(axes[:8], feat_cols):
    sns.histplot(possible_cohorts[feat].dropna(), bins=40, ax=ax)
    ax.set_title(feat)

sns.scatterplot(data=possible_cohorts, x="n_cohort", y="target_rate", s=10, alpha=0.4, ax=axes[8])
sns.scatterplot(data=possible_cohorts, x="log_n_cohort", y="log_target_rate", s=10, alpha=0.4, ax=axes[9])
sns.scatterplot(data=possible_cohorts, x="log_n_cohort", y="log_target_rate", hue="D1cat", palette=cmap, alpha=0.4, legend=False, ax=axes[10])
sns.scatterplot(data=possible_cohorts, x="log_n_cohort", y="log_target_rate", hue="D2cat", palette=cmap, alpha=0.4, legend=False, ax=axes[11])
axes[10].set_title("D1")
axes[11].set_title("D2")

plt.tight_layout()
plt.savefig(save_dir + "cohort_characteristics.png", dpi=150)
plt.show()



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

d1 = possible_cohorts["D1type"].value_counts()
d2 = possible_cohorts["D2type"].value_counts()

types = sorted(set(d1.index) | set(d2.index))

# order by max(D1, D2) descending
order = (
    pd.DataFrame({"D1": d1.reindex(types).fillna(0), "D2": d2.reindex(types).fillna(0)})
    .assign(max_=lambda x: x[["D1", "D2"]].max(axis=1))
    .sort_values("max_", ascending=False)
    .index.tolist()
)

D1 = d1.reindex(order).fillna(0).values
D2 = d2.reindex(order).fillna(0).values

y = np.arange(len(order))
h = 0.4  # bar height

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(y - h/2, D1, height=h, color="#A6CEE3", label="D1")
ax.barh(y + h/2, D2, height=h, color="#FDBF6F", label="D2")

ax.set_yticks(y)
ax.set_yticklabels(order)
ax.invert_yaxis()  # largest at top
ax.set_xlabel("count")
ax.set_title(f"Top type counts (D1 vs D2)")
ax.legend()
plt.tight_layout()
plt.savefig(save_dir + "DTB_cohorts_D1_D2_counts.png", dpi=150)
plt.show()

In [ ]:
D1_to_D2 = (
    possible_cohorts.groupby("D1type")["D2type"]
    .value_counts()
    .groupby(level=0)
    .head(5)
    .reset_index(name="count")
)
D1_to_D2.sort_values("count", ascending=False)

k = 5
common = (
    possible_cohorts.groupby("D1type")["D2type"]
    .value_counts()
    .groupby(level=0).head(k)
    .rename("count")
    .reset_index()
)

pivot = common.pivot(index="D1type", columns="D2type", values="count").fillna(0)
colors = sns.color_palette("tab20") 
ax = pivot.plot(kind="bar", stacked=True, figsize=(14,6),color=colors)
ax.set_title(f"Top {k} D2type per D1type")
ax.set_xlabel("D1type")
ax.set_ylabel("count")
plt.legend(title="D2type", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## UMAP

In [ ]:
feat_cols = ["log_n_cohort", "target_rate", "log_RR"]
df_feat = possible_cohorts[feat_cols + ["D1", "D2", "D1type", "D2type"]].dropna().copy()

X = StandardScaler().fit_transform(df_feat[feat_cols])
reducer = umap.UMAP(n_components=2, random_state=42)
coords = reducer.fit_transform(X)
df_feat["U1"] = coords[:, 0]
df_feat["U2"] = coords[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(18, 5))
for ax, col in zip(axes, ["log_n_cohort", "target_rate"]):
    sc = ax.scatter(df_feat["U1"], df_feat["U2"], c=df_feat[col], cmap="viridis", s=5, alpha=0.5)
    plt.colorbar(sc, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.savefig(save_dir + "umap_features.png", dpi=150)
plt.show()

## K-means — elbow + silhouette

In [ ]:

X = StandardScaler().fit_transform(df_feat[feat_cols])

inertias, silhouettes = [], []
n_clusters = range(2, 11)

for k in n_clusters:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(n_clusters, inertias, marker="o")
axes[0].set_xlabel("k")
axes[0].set_title("Elbow")
axes[1].plot(n_clusters, silhouettes, marker="o")
axes[1].set_xlabel("k")
axes[1].set_title("Silhouette")
plt.tight_layout()
plt.savefig(save_dir + "kmeans_selection.png", dpi=150)
plt.show()

## Apply clustering and describe clusters

In [ ]:

k = 3  # selected based on the above cell
km = KMeans(n_clusters=k, random_state=42, n_init=10)
df_feat["cluster"] = km.fit_predict(X)

fig, ax = plt.subplots(figsize=(8, 6))
palette = sns.color_palette("tab10", k)
for c in range(k):
    sub = df_feat[df_feat["cluster"] == c]
    ax.scatter(sub["U1"], sub["U2"], label=f"Cluster {c}", color=palette[c], s=5, alpha=0.5)
ax.legend(markerscale=3)
ax.set_title("UMAP colored by cluster")
plt.tight_layout()
plt.savefig(save_dir + "umap_clusters.png", dpi=150)
plt.show()

df_feat[feat_cols + ["cluster"]].groupby("cluster").mean().round(3)

## Cluster entropy and heterogeneity

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, feat in zip(axes, ["target_rate", "log_n_cohort", "log_RR"]):
    sns.boxplot(data=df_feat, x="cluster", y=feat, palette="tab10", ax=ax)
    ax.set_title(feat)
plt.tight_layout()
plt.savefig(save_dir + "cluster_distributions.png", dpi=150)#

plt.show()


In [ ]:
print("Within-cluster std (heterogeneity):")
df_feat.groupby("cluster")[feat_cols].std().round(3)

## Disease type distribution per cluster

In [ ]:
ct = pd.crosstab(df_feat["cluster"], df_feat["D1type"], normalize="index")

ct.plot(kind="bar", stacked=True, figsize=(12, 5), colormap="tab20")
plt.title("D1type distribution per cluster (normalized)")
plt.ylabel("proportion")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(save_dir + "cluster_D1type.png", dpi=150)
plt.show()

## Stratified sampling — representative cohorts

In [ ]:
# Based on clustering (LATER)

'''N_PER_CLUSTER = 10

sampled = []
for c in df_feat["cluster"].unique():
    pool = df_feat[df_feat["cluster"] == c]
    n = min(N_PER_CLUSTER, len(pool))
    sampled.append(pool.sample(n=n, random_state=42))

sampled_df = pd.concat(sampled)
rep_cohorts = df.loc[sampled_df.index].copy()
rep_cohorts["cluster"] = sampled_df["cluster"]
print(f"\nTotal sampled: {len(rep_cohorts)}")

out_path = os.path.join(save_dir, "representative_cohorts.csv")
rep_cohorts.to_csv(out_path, index=False)
print(f"Saved to {out_path}")
'''

In [ ]:
# assign bins based on fix intervals
target_metrics = ["target_rate", "log_n_cohort"]

bin_labels = ["very_low", "low", "medium", "high"]
for target_metric, col_name in [("log_target_rate", "tr_group"), ("log_n_cohort", "cs_group")]:
    lo, hi = possible_cohorts[target_metric].min(), possible_cohorts[target_metric].max()
    step = (hi - lo) / 4
    bins = [lo + i * step for i in range(5)]
    possible_cohorts[col_name] = pd.cut(possible_cohorts[target_metric], bins=bins, labels=bin_labels, include_lowest=True)

In [ ]:
## select metric
metric = "tr_group" # tr_group or cs_group 

high_cohort = possible_cohorts[possible_cohorts[metric] == "high"].head(10)
medium_cohort = possible_cohorts[possible_cohorts[metric] == "medium"].head(10)
low_cohort = possible_cohorts[possible_cohorts[metric] == "low"].head(10)
very_low_cohort = possible_cohorts[possible_cohorts[metric] == "very_low"].head(10)


# Build all cohorts df
selected_cohorts = pd.concat([high_cohort, medium_cohort, low_cohort, very_low_cohort])

## ML Results

In [ ]:
from pathlib import Path
model ="CatBoost"
results_df = pd.DataFrame()
for cohort in selected_cohorts["cohort_name"]:
    file = dir = Path(f"/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/flabnet-pipeline/MIMIC_IV/saved_data/results/{cohort}/non_time_series/concatenate/minority/feature_selection_True/grid_False/feat_type_VMD/agg_interval_24h/2503/CatBoost_results.csv")
    if file.exists():
        output = pd.read_csv(file)
        output["cohort_name"] = cohort 
        results_df = pd.concat([results_df, output])
        
        
        
results_df =results_df[["cohort_name","Fold","AUC-ROC", "AUC-PRC", "f1_score"]]
for col in ["AUC-ROC", "AUC-PRC", "f1_score"]:
    results_df[col] = pd.to_numeric(results_df[col], errors="coerce")

for col, new_col in [("AUC-ROC", "roc_avg"), ("AUC-PRC", "pr_avg"), ("f1_score", "f1_avg")]:
    results_df[new_col] = results_df.groupby("cohort_name")[col].transform("mean")

results_df = results_df[["cohort_name", "roc_avg", "pr_avg", "f1_avg"]].drop_duplicates("cohort_name")
results_df =results_df.merge(selected_cohorts[["cohort_name","D1", "D2"]], on=["cohort_name"], how="inner")
results_df =results_df.merge(selected_cohorts[["D1", "D2", "target_rate", "n_cohort", "tr_group", "cs_group","RR", "female_counts", "log_imb"]], on=["D1", "D2"], how="inner")

results_df.sort_values(["roc_avg"], ascending=False)
results_df.to_csv("/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/FLabBench-pipeline/saved_data/plots/DTB/selected_cohorts_results.csv",index=False)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


df_merged = results_df.copy()
metric = "roc_avg"
group_col = "tr_group"
order = ["low", "medium", "high"]
palette = dict(zip(order, sns.color_palette("Set2", len(order))))

fig, ax = plt.subplots(figsize=(8, 5))
sns.stripplot(
    data=df_merged.dropna(subset=[metric]),
    x=group_col, y=metric,
    order=order, palette=palette,
    jitter=0.25, alpha=0.5, size=4, ax=ax
)
sns.boxplot(
    data=df_merged.dropna(subset=[metric]),
    x=group_col, y=metric,
    order=order, palette=palette,
    width=0.4, fliersize=0, ax=ax
)
#ax.axhline(0.5, color="gray", linestyle="--", linewidth=1, label="random baseline")
ax.set_xlabel("Cohort size group", fontsize=11)
ax.set_ylabel(metric, fontsize=11)
ax.set_title("Model performance", fontsize=11)
ax.legend()
plt.tight_layout()
plt.savefig(save_dir + f"DTB_cohorts_{metric}_by_{group_col}.png", dpi=150)
plt.show()

## statistical analysis


In [ ]:
from scipy import stats
import scikit_posthocs as sp

metric = "pr_avg"
group_col = "tr_group"


data = df_merged.dropna(subset=["roc_avg"])
groups = [grp["roc_avg"].values for _, grp in data.groupby(group_col)]

stat, p = stats.kruskal(*groups)
dunn = sp.posthoc_dunn(data, val_col=metric, group_col=group_col, p_adjust="fdr_bh")

print(f"Kruskal-Wallis: H={stat:.3f}, p={p:.4f}")
print("\nDunn post-hoc p-values (BH corrected):")
print(dunn.round(4))

print("\nSignificantly different pairs (p < 0.05):")
order = ["very_low","low", "medium", "high"]
for i in range(len(order)):
    for j in range(i+1, len(order)):
        g1, g2 = order[i], order[j]
        pval = dunn.loc[g1, g2]
        sig = "YES ***" if pval < 0.001 else ("YES *" if pval < 0.05 else "no")
        print(f"  {g1} vs {g2}: p={pval:.4f}  →  {sig}")

## Feature analysis

In [ ]:
from tqdm import tqdm
from scipy.stats import entropy

import pickle
with open("/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/flabnet-pipeline/MIMIC_IV/saved_data/top_features/mimic_top100_features.pkl", "rb") as f:
    top_features = pickle.load(f)
top_features = set(int(x) for x in top_features)

results = pd.read_csv("/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/FLabBench-pipeline/saved_data/plots/DTB/selected_cohorts_results.csv")

cohorts = results.cohort_name


def feature_entropy(series):
    series = series.dropna()
    if series.std() > 0:
        series = (series - series.mean()) / series.std()
    counts, _ = np.histogram(series, bins=10)
    return entropy(counts + 1)


for cohort_name in tqdm(cohorts):
    cohort = pd.read_csv(f"/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/flabnet-pipeline/MIMIC_IV/saved_data/cohorts/{cohort_name}.csv")
    features = pd.read_csv(f"/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/flabnet-pipeline/MIMIC_IV/saved_data/features/{cohort_name}_admissions_labs_14_days.csv.gz")
    features = features[features["itemid"].isin(list(top_features)[:100])]
    features = features.merge(cohort[["hadm_id", "label"]], on="hadm_id")
    patient_feats = features.groupby(["hadm_id", "itemid"])["value"].mean().unstack()
    labels = cohort[["hadm_id", "label"]].drop_duplicates("hadm_id")
    patient_feats = patient_feats.merge(labels, on="hadm_id")
    missingness_var = patient_feats.drop(columns="label").isna().mean().var() # variation of missingness across features 
    results.loc[results["cohort_name"] == cohort_name, "missingness_v"] = missingness_var
    
    missingness_mean = patient_feats.drop(columns="label").isna().mean().mean() # overall average missingness rate
    results.loc[results["cohort_name"] == cohort_name, "missingness_m"] = missingness_mean
    
    pos = patient_feats[patient_feats["label"] == 1].drop(columns="label").mean()
    neg = patient_feats[patient_feats["label"] == 0].drop(columns="label").mean()
    
    feat_cols = patient_feats.drop(columns="label").columns
    h_pos = patient_feats[patient_feats["label"]==1][feat_cols].apply(feature_entropy).mean()
    h_neg = patient_feats[patient_feats["label"]==0][feat_cols].apply(feature_entropy).mean()
    
    results.loc[results["cohort_name"] == cohort_name, "h_pos"] = h_pos
    results.loc[results["cohort_name"] == cohort_name, "h_neg"] = h_neg
    

    pooled_std = patient_feats.drop(columns="label").std()
    cohens_d = ((pos - neg).abs() / (pooled_std + 1e-8)).mean()

    results.loc[results["cohort_name"] == cohort_name, "cohens_d"] = cohens_d

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(12, 8))
for ax, feat in zip(axes.ravel(), ["n_cohort", "target_rate", "missingness_m","missingness_v", "h_pos", "h_neg","log_imb", "RR"]): # female_counts
    sns.scatterplot(data=results, x=feat, y="roc_avg", ax=ax)
    ax.set_title(feat)
plt.tight_layout()
plt.savefig(save_dir + "features.png", dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(18, 6))
top_feats = patient_feats.drop(columns="label").columns[1:21]

for ax, col in zip(axes.ravel(), top_feats):
    patient_feats[patient_feats["label"]==1][col].dropna().hist(ax=ax, bins=20, alpha=0.5, label="pos", color="red",density=True)
    patient_feats[patient_feats["label"]==0][col].dropna().hist(ax=ax, bins=20, alpha=0.5, label="neg", color="blue",density=True)
    ax.set_title(col)
    ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.decomposition import PCA

X = patient_feats.drop(columns="label").fillna(0)
X.columns = X.columns.astype(str)
coords = PCA(n_components=2).fit_transform(StandardScaler().fit_transform(X))
plt.scatter(coords[:,0], coords[:,1], c=patient_feats["label"], cmap="coolwarm", alpha=1, s=10)
plt.colorbar(label="label")
plt.title("PCA: pos vs neg patients")
plt.show()

# CHECK LATER

# Find largest Cohort

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

root_dir = "/Users/zy51nise/Documents/BIONETs/FLabNet/Code/FLabBench-pipeline/data/MIMIC_IV/cohorts/DTB/"
sel_edges = pd.read_csv(root_dir + "selected_edges_DTB_all.csv")


possible_cohorts = sel_edges[(sel_edges["n_pos"] > 10) & (sel_edges["n_neg"] > 50)]
possible_cohorts["n_cohort"] = possible_cohorts["n_pos"] + possible_cohorts["n_neg"]
possible_cohorts["target_rate"] = possible_cohorts["n_pos"] / possible_cohorts["n_cohort"]
possible_cohorts["log_n_cohort"] = np.log(possible_cohorts["n_cohort"])
possible_cohorts["log_target_rate"] = np.log(possible_cohorts["target_rate"])

sub_cohorts =possible_cohorts[possible_cohorts['target_rate'] > 0.01]
sub_cohorts.sort_values(by='n_cohort', ascending=False).head(5)


In [ ]:
cohort = pd.read_csv(root_dir + "cohort_E78_H35_791.08.csv.gz", compression="gzip")
print(cohort["label"].sum())
print(len(cohort))